In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import pickle

In [4]:
df = pd.read_csv(r"/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/all_stations_combined.csv")
print(df.columns.tolist())
print(df.head(3))

['date;station_name;lat;lon;height_m;TG;TN;TNH;TX;TXH;T10N;T10NH;TZG;UG;UN;UNH;UX;UXH;EEG;RH;RHX;RHXH;DR;FG;DDVEC;FHVEC;FHN;FHNH;FHX;FHXH;DDFHX;FXX;FXXH;Q;SQ;SP;EV24;PG;PN;PNH;PX;PXH;VVN;VVNH;VVX;VVXH;W1;W2;W3;W5;W6;NG']
  date;station_name;lat;lon;height_m;TG;TN;TNH;TX;TXH;T10N;T10NH;TZG;UG;UN;UNH;UX;UXH;EEG;RH;RHX;RHXH;DR;FG;DDVEC;FHVEC;FHN;FHNH;FHX;FHXH;DDFHX;FXX;FXXH;Q;SQ;SP;EV24;PG;PN;PNH;PX;PXH;VVN;VVNH;VVX;VVXH;W1;W2;W3;W5;W6;NG
0  02/01/1975;AMSTERDAM/SCHIPHOL AP;52.3172;4.789...                                                                                                                                                                      
1  02/01/1975;CADZAND WP;51.38;3.3792;0;;;;;;;;;;...                                                                                                                                                                      
2  02/01/1975;DE BILT AWS;52.0989;5.1797;1.9;7.9;...                                                                      

In [5]:
# ── 1. Load CSV ───────────────────────────────────────────────────────────────
df = pd.read_csv(
    r"/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/all_stations_combined.csv",
    sep=";",
    parse_dates=["date"],
    dayfirst=True                # because your dates are dd/mm/yyyy
)
df = df.sort_values(["date", "station_name"]).reset_index(drop=True)

In [6]:
# ── 1b. Filter to NL stations only ───────────────────────────────────────────
NL_LAT = (50.75, 53.75)
NL_LON = (3.20,  7.25)
before = df["station_name"].nunique()
df = df[df["lat"].between(*NL_LAT) & df["lon"].between(*NL_LON)].reset_index(drop=True)
print(f"After NL filter     : {before} → {df['station_name'].nunique()} stations")

After NL filter     : 123 → 114 stations


In [7]:
# ── 1c. Deduplicate stations ──────────────────────────────────────────────────
duplicate_groups = [
    ["AMSTERDAM/SCHIPHOL AP", "Schiphol Airport"],
    ["ARCEN AWS", "Arcen"], ["BERKHOUT AWS", "Berkhout"],
    ["CABAUW TOWER AWS", "Cabauw"], ["DE KOOY VK", "De Kooy Airport"],
    ["DE BILT AWS", "De Bilt"], ["CADZAND WP", "Cadzand"],
    ["HERWIJNEN AWS", "Herwijnen"], ["GRONINGEN AP EELDE", "Groningen Airport Eelde"],
    ["HANSWEERT", "Hansweert"], ["HEINO AWS", "Heino"],
    ["GILZE RIJEN", "Gilze-Rijen Airport"], ["EINDHOVEN AP", "Eindhoven Airport"],
    ["ELL AWS", "Ell"], ["EURO PLATFORM", "Europlatform"],
    ["DEELEN", "Deelen Airport"], ["Hollandse Kust Zuid Alfa (HKZA)", "Hollandse Kust Zuid Alpha"],
    ["Hoorn Terschelling", "TERSCHELLING HOORN AWS"], ["IJMUIDEN WP", "IJmuiden"],
    ["LELYSTAD AP", "Lelystad Airport"], ["LEEUWARDEN", "Leeuwarden Airport"],
    ["LAUWERSOOG AWS", "Lauwersoog"], ["LICHTEILAND GOEREE", "Lichteiland Goeree"],
    ["MAASTRICHT AACHEN AP", "Maastricht Airport"], ["HOUTRIBDIJK WP", "Houtribdijk"],
    ["HUIBERTGAT WP", "Huibertgat"], ["IJMOND WP", "IJmond"],
    ["HUPSEL AWS", "Hupsel"], ["HOOGEVEEN AWS", "Hoogeveen"],
    ["HOEK VAN HOLLAND AWS", "Hoek van Holland"], ["MAROLLEGAT", "Tholen"],
    ["MARKNESSE AWS", "Marknesse"], ["VOLKEL", "Volkel Airport"],
    ["VOORSCHOTEN AWS", "Voorschoten"], ["TEXELHORS WP", "Texelhors"],
    ["STAVOREN AWS", "Stavoren"], ["VLAKTE VAN DE RAAN", "Vlakte van de Raan"],
    ["VLIELAND", "Vlieland Vliehors"], ["VLISSINGEN AWS", "Vlissingen"],
    ["TWENTHE AWS", "Twenthe Airport"], ["STAVENISSE", "Stavenisse"],
    ["OOSTERSCHELDE WP", "Oosterschelde"], ["ROTTERDAM THE HAGUE AP", "Rotterdam Airport"],
    ["WESTDORPE AWS", "Westdorpe"], ["WIJK AAN ZEE AWS", "Wijk aan Zee"],
    ["WILHELMINADORP AWS", "Wilhelminadorp"], ["WOENSDRECHT", "Woensdrecht Airport"],
    ["WIJDENES WP", "Wijdenes"], ["NIEUW BEERTA AWS", "Nieuw Beerta"],
    ["OOSTERSCHELDE 4", "Schaar"], ["ROTTERDAM GEULHAVEN", "Rotterdam Geulhaven"],
]

to_remove = []
for group in duplicate_groups:
    present = [s for s in group if s in df["station_name"].values]
    if len(present) < 2:
        continue
    counts = {s: df[df["station_name"] == s]["RH"].count() for s in present}
    to_remove.append(min(counts, key=counts.get))

# Handle VALKENBURG VK vs VOORSCHOTEN AWS
if all(s in df["station_name"].values for s in ["VALKENBURG VK", "VOORSCHOTEN AWS"]):
    counts = {s: df[df["station_name"] == s]["RH"].count() for s in ["VALKENBURG VK", "VOORSCHOTEN AWS"]}
    to_remove.append(min(counts, key=counts.get))

df = df[~df["station_name"].isin(to_remove)].reset_index(drop=True)
print(f"After dedup         : {df['station_name'].nunique()} stations")

After dedup         : 62 stations


In [8]:
# ── Verify no duplicates remain ───────────────────────────────────────────────
from sklearn.metrics.pairwise import haversine_distances

stations_check = df.groupby("station_name")[["lat", "lon"]].first().reset_index()
coords_rad = np.radians(stations_check[["lat", "lon"]].values)
dist_matrix = haversine_distances(coords_rad) * 6371

pairs = []
for i in range(len(stations_check)):
    for j in range(i+1, len(stations_check)):
        if dist_matrix[i, j] < 5:                       # within 5km
            pairs.append({
                "station_a": stations_check.iloc[i]["station_name"],
                "station_b": stations_check.iloc[j]["station_name"],
                "dist_km"  : round(dist_matrix[i, j], 2)
            })

if pairs:
    print("Remaining near-duplicates (<5km):")
    print(pd.DataFrame(pairs).to_string(index=False))
else:
    print(f"All {len(stations_check)} stations are unique — no duplicates within 5km!")
    print("\nProceed with the full pipeline.")

Remaining near-duplicates (<5km):
station_a station_b  dist_km
   IJmond  IJmuiden     2.55


In [9]:
# Keep whichever has more RH observations
counts = {s: df[df["station_name"] == s]["RH"].count() for s in ["VALKENBURG VK", "VOORSCHOTEN AWS"]}
print(counts)

worst = min(counts, key=counts.get)
df = df[df["station_name"] != worst].reset_index(drop=True)

print(f"Removed: {worst}")
print(f"Stations remaining: {df['station_name'].nunique()}")

{'VALKENBURG VK': np.int64(15097), 'VOORSCHOTEN AWS': np.int64(0)}
Removed: VOORSCHOTEN AWS
Stations remaining: 62


In [10]:
# ── 1d. Trim time range to 1995 onwards ──────────────────────────────────────
START_DATE = "1995-01-01"
df = df[df["date"] >= START_DATE].reset_index(drop=True)
print(f"After time trim     : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Days                : {df['date'].nunique()}")

# ── 1e. Drop stations with <20% RH coverage ───────────────────────────────────
rh_pivot = df.pivot(index="date", columns="station_name", values="RH")
coverage = rh_pivot.notna().sum() / len(rh_pivot) * 100
low_cov  = coverage[coverage < 20].index.tolist()
df = df[~df["station_name"].isin(low_cov)].reset_index(drop=True)
print(f"After RH filter     : {df['station_name'].nunique()} stations  (dropped {len(low_cov)} low-coverage)")


After time trim     : 1995-01-01 → 2026-03-13
Days                : 11395
After RH filter     : 34 stations  (dropped 28 low-coverage)


In [11]:
print (df.columns.tolist())
print (df.head(3))

['date', 'station_name', 'lat', 'lon', 'height_m', 'TG', 'TN', 'TNH', 'TX', 'TXH', 'T10N', 'T10NH', 'TZG', 'UG', 'UN', 'UNH', 'UX', 'UXH', 'EEG', 'RH', 'RHX', 'RHXH', 'DR', 'FG', 'DDVEC', 'FHVEC', 'FHN', 'FHNH', 'FHX', 'FHXH', 'DDFHX', 'FXX', 'FXXH', 'Q', 'SQ', 'SP', 'EV24', 'PG', 'PN', 'PNH', 'PX', 'PXH', 'VVN', 'VVNH', 'VVX', 'VVXH', 'W1', 'W2', 'W3', 'W5', 'W6', 'NG']
        date           station_name      lat     lon  height_m   TG   TN  \
0 1995-01-01  AMSTERDAM/SCHIPHOL AP  52.3172  4.7897     -3.87  5.9  2.0   
1 1995-01-01              ARCEN AWS  51.4972  6.1961     19.50  4.7  2.0   
2 1995-01-01       CABAUW TOWER AWS  51.9692  4.9258     -0.71  4.7  1.0   

    TNH   TX   TXH  ...   VVN  VVNH   VVX  VVXH   W1    W2   W3   W5  W6   NG  
0  22.0  7.9   1.0  ...  60.0  22.0  75.0   6.0  0.0  12.0  1.0  0.0 NaN  5.0  
1  24.0  7.0  13.0  ...   NaN   NaN   NaN   NaN  NaN   NaN  NaN  NaN NaN  NaN  
2  22.0  7.2   4.0  ...   NaN   NaN   NaN   NaN  NaN   NaN  NaN  NaN NaN  NaN  



In [12]:
# ── 2. Auto-detect feature columns ───────────────────────────────────────────
non_feature_cols = {"date", "station_name", "station", "wsi", "lat", "lon", "height_m"}
feature_cols = [c for c in df.columns if c not in non_feature_cols]
print(f"Features detected   : {len(feature_cols)} {feature_cols}")

Features detected   : 47 ['TG', 'TN', 'TNH', 'TX', 'TXH', 'T10N', 'T10NH', 'TZG', 'UG', 'UN', 'UNH', 'UX', 'UXH', 'EEG', 'RH', 'RHX', 'RHXH', 'DR', 'FG', 'DDVEC', 'FHVEC', 'FHN', 'FHNH', 'FHX', 'FHXH', 'DDFHX', 'FXX', 'FXXH', 'Q', 'SQ', 'SP', 'EV24', 'PG', 'PN', 'PNH', 'PX', 'PXH', 'VVN', 'VVNH', 'VVX', 'VVXH', 'W1', 'W2', 'W3', 'W5', 'W6', 'NG']


In [13]:
# ── 3. Station metadata ───────────────────────────────────────────────────────
stations = (
    df.groupby("station_name")[["lat", "lon", "height_m"]]
    .first()
    .reset_index()
    .reset_index(drop=False)
    .rename(columns={"index": "node_id"})
)
N = len(stations)
print(f"Stations: {N}")

Stations: 34


In [14]:
# ── 4. Build KNN edges from lat/lon ───────────────────────────────────────────
K = 5
coords = stations[["lat", "lon"]].values
nbrs = NearestNeighbors(n_neighbors=K + 1).fit(coords)
distances, indices = nbrs.kneighbors(coords)

rows, cols, edge_weights = [], [], []
for i, (dists, neighbours) in enumerate(zip(distances, indices)):
    for d, j in zip(dists[1:], neighbours[1:]):
        rows.append(i)
        cols.append(j)
        edge_weights.append(1.0 / (d + 1e-6))

edge_index = np.array([rows, cols])               # [2, num_edges]
edge_attr  = np.array(edge_weights)               # [num_edges]
print(f"Edges: {edge_index.shape[1]}  (K={K} neighbours per station)")

Edges: 170  (K=5 neighbours per station)


In [15]:
# ── 5. Pivot to [T, N, F] array ───────────────────────────────────────────────
station_order = stations["station_name"].tolist()

pivots = []
for feat in feature_cols:
    p = (
        df.pivot(index="date", columns="station_name", values=feat)
        .sort_index()
        .reindex(columns=station_order)
    )
    pivots.append(p.values)

dates      = df.pivot(index="date", columns="station_name", values="RH").sort_index().index
data_array = np.stack(pivots, axis=-1).astype(np.float32)
T, _, F    = data_array.shape
print(f"Array shape         : {data_array.shape}  [T, N, F]")

Array shape         : (10897, 34, 47)  [T, N, F]


In [16]:
# ── 6. Drop high-NaN features (>80% missing) ─────────────────────────────────
nan_per_feat = [np.isnan(data_array[:, :, i]).mean() * 100 for i in range(F)]
keep_idx     = [i for i, pct in enumerate(nan_per_feat) if pct <= 80]
drop_feats   = [feature_cols[i] for i, pct in enumerate(nan_per_feat) if pct > 80]
print(f"Dropping high-NaN features (>80%): {drop_feats}")

data_array   = data_array[:, :, keep_idx]
feature_cols = [feature_cols[i] for i in keep_idx]
F            = len(feature_cols)
print(f"Features remaining  : {F}")

Dropping high-NaN features (>80%): ['TZG']
Features remaining  : 46


In [17]:
# ── 7. NaN summary ────────────────────────────────────────────────────────────
# ── 8. NaN summary ────────────────────────────────────────────────────────────
# Now reflects raw (un-normalised) values — that is correct for the base file.
# No changes needed.
overall_nan = np.isnan(data_array).mean() * 100
print(f"\nOverall NaN         : {overall_nan:.2f}%")
for i, feat in enumerate(feature_cols):
    pct = np.isnan(data_array[:, :, i]).mean() * 100
    if pct > 0:
        print(f"  {feat:<10}  {pct:.1f}% missing")


Overall NaN         : 16.70%
  TG          4.5% missing
  TN          4.6% missing
  TNH         4.6% missing
  TX          4.6% missing
  TXH         4.6% missing
  T10N        7.2% missing
  T10NH       7.2% missing
  UG          4.5% missing
  UN          4.5% missing
  UNH         4.5% missing
  UX          4.5% missing
  UXH         4.5% missing
  EEG         4.5% missing
  RH          6.8% missing
  RHX         6.8% missing
  RHXH        6.8% missing
  DR          6.9% missing
  FG          6.9% missing
  DDVEC       6.9% missing
  FHVEC       6.9% missing
  FHN         6.9% missing
  FHNH        6.9% missing
  FHX         6.9% missing
  FHXH        6.9% missing
  DDFHX       6.9% missing
  FXX         7.0% missing
  FXXH        7.0% missing
  Q           7.4% missing
  SQ          7.6% missing
  SP          7.7% missing
  EV24        7.4% missing
  PG          36.4% missing
  PN          36.4% missing
  PNH         36.4% missing
  PX          36.4% missing
  PXH         36.4% m

In [19]:
# ── Run this before the save cell ────────────────────────────────────────────
rh_idx = feature_cols.index("RH")
print(f"RH index: {rh_idx}")   # should print 13

RH index: 13


In [29]:
# ── 9. Save base pkl ──────────────────────────────────────────────────────────
# Three changes vs your original:
#   1. New filename: gnn_base.pkl (never overwrite this again)
#   2. data_array is RAW — no normalisation applied
#   3. No scalers key — scalers are built at training time by build_transforms()

base_path = r"/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_base.pkl"

with open(base_path, "wb") as f:
    pickle.dump({
        "data_array"   : data_array,    # raw, un-normalised [T, N, F]
        "dates"        : dates,
        "stations"     : stations,
        "feature_cols" : feature_cols,
        "rh_idx"       : rh_idx,        # index of RH in feature_cols
        "edge_index"   : edge_index,
        "edge_attr"    : edge_attr,
        "K"            : K,
        # NO scalers — normalisation happens in training notebook
    }, f)

print(f"Base pkl saved to {base_path}")
print(f"Shape : {data_array.shape}  [T, N, F]")
print(f"This file never needs to be re-run unless raw data changes.")

Base pkl saved to /media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_base.pkl
Shape : (10897, 34, 46)  [T, N, F]
This file never needs to be re-run unless raw data changes.


Now we check the pickle datasets which is Python's way of saving any object (arrays, lists, dictionaries) to disk in binary format so you can reload it later exactly as it was.

In [30]:
with open(r"/media/user/DataDisk/Python/Dordrecht/AI Model/GNN-model/gnn_base.pkl", "rb") as f:
    prepared = pickle.load(f)

In [32]:
# ── Contents ──────────────────────────────────────────────────────────────────
print("Keys saved:", list(prepared.keys()))

data_array   = prepared["data_array"]
dates        = prepared["dates"]
stations     = prepared["stations"]
feature_cols = prepared["feature_cols"]
rh_idx       = prepared["rh_idx"]
edge_index   = prepared["edge_index"]
edge_attr    = prepared["edge_attr"]


Keys saved: ['data_array', 'dates', 'stations', 'feature_cols', 'rh_idx', 'edge_index', 'edge_attr', 'K']


In [33]:
# ── Shapes ────────────────────────────────────────────────────────────────────
print(f"\n── Array shapes ──────────────────────────────")
print(f"data_array : {data_array.shape}   ← [T, N, F]")
print(f"edge_index : {edge_index.shape}   ← [2, num_edges]")
print(f"edge_attr  : {edge_attr.shape}    ← [num_edges]")



── Array shapes ──────────────────────────────
data_array : (10897, 34, 46)   ← [T, N, F]
edge_index : (2, 170)   ← [2, num_edges]
edge_attr  : (170,)    ← [num_edges]


In [34]:
# ── Time range ────────────────────────────────────────────────────────────────
print(f"\n── Time range ────────────────────────────────")
print(f"From : {dates[0]}")
print(f"To   : {dates[-1]}")
print(f"Days : {len(dates)}")


── Time range ────────────────────────────────
From : 1995-01-01 00:00:00
To   : 2024-10-31 00:00:00
Days : 10897


In [35]:
# ── Stations ──────────────────────────────────────────────────────────────────
print(f"\n── Stations ({len(stations)}) ────────────────────────────")
print(stations[["node_id", "station_name", "lat", "lon"]].to_string(index=False))


── Stations (34) ────────────────────────────
 node_id           station_name     lat    lon
       0  AMSTERDAM/SCHIPHOL AP 52.3172 4.7897
       1              ARCEN AWS 51.4972 6.1961
       2           BERKHOUT AWS 52.6428 4.9789
       3       CABAUW TOWER AWS 51.9692 4.9258
       4            DE BILT AWS 52.0989 5.1797
       5             DE KOOY VK 52.9269 4.7811
       6                 DEELEN 52.0547 5.8722
       7           EINDHOVEN AP 51.4497 5.3769
       8                ELL AWS 51.1967 5.7625
       9            GILZE RIJEN 51.5650 4.9353
      10     GRONINGEN AP EELDE 53.1236 6.5847
      11              HEINO AWS 52.4344 6.2589
      12          HERWIJNEN AWS 51.8578 5.1453
      13   HOEK VAN HOLLAND AWS 51.9911 4.1217
      14          HOOGEVEEN AWS 52.7489 6.5731
      15             HUPSEL AWS 52.0678 6.6567
      16         LAUWERSOOG AWS 53.4117 6.1992
      17             LEEUWARDEN 53.2231 5.7517
      18            LELYSTAD AP 52.4483 5.5081
      19   MA

In [38]:
# ── NaN check ─────────────────────────────────────────────────────────────────
print(f"\n── NaN check ─────────────────────────────────")
nan_pct = np.isnan(data_array).mean() * 100
print(f"Overall NaN: {nan_pct:.2f}%")
for i, feat in enumerate(feature_cols):
    pct = np.isnan(data_array[:, :, i]).mean() * 100
    if pct > 0:
        print(f"  {feat:<10}  {pct:.1f}% missing")


── NaN check ─────────────────────────────────
Overall NaN: 16.70%
  TG          4.5% missing
  TN          4.6% missing
  TNH         4.6% missing
  TX          4.6% missing
  TXH         4.6% missing
  T10N        7.2% missing
  T10NH       7.2% missing
  UG          4.5% missing
  UN          4.5% missing
  UNH         4.5% missing
  UX          4.5% missing
  UXH         4.5% missing
  EEG         4.5% missing
  RH          6.8% missing
  RHX         6.8% missing
  RHXH        6.8% missing
  DR          6.9% missing
  FG          6.9% missing
  DDVEC       6.9% missing
  FHVEC       6.9% missing
  FHN         6.9% missing
  FHNH        6.9% missing
  FHX         6.9% missing
  FHXH        6.9% missing
  DDFHX       6.9% missing
  FXX         7.0% missing
  FXXH        7.0% missing
  Q           7.4% missing
  SQ          7.6% missing
  SP          7.7% missing
  EV24        7.4% missing
  PG          36.4% missing
  PN          36.4% missing
  PNH         36.4% missing
  PX       

In [39]:
# Check for stations with identical coordinates
dupes = stations[stations.duplicated(subset=["lat", "lon"], keep=False)]
print(f"Stations sharing coordinates: {len(dupes)}")
print(dupes[["node_id", "station_name", "lat", "lon"]].sort_values(["lat", "lon"]).to_string(index=False))

Stations sharing coordinates: 0
Empty DataFrame
Columns: [node_id, station_name, lat, lon]
Index: []
